# Mario Kart Tracker v2: Data Scraping

## Characters

The aim of this notebook is to scrape the characters in both Mario Kart 8 Deluxe and Mario Kart World from the internet and store them as a CSV file

Is web scraping overkill? Yes. But we're learning!

Website to scrape from: Mario Wiki

Required libraries:
* `Beautiful Soup`
* `requests` 
* `Selenium`

In [1]:
from bs4 import BeautifulSoup
import requests
from selenium import webdriver
import pandas as pd
from pathlib import Path

In [2]:
wd = Path().cwd()
wd = wd.parent

The processes for each will be slightly different, as the layouts are different. Let's start with MK World

In [ ]:
url = "https://mario.fandom.com/wiki/List_of_characters_in_Mario_Kart_World"

driver = webdriver.Firefox()
driver.get(url)

world_html = driver.page_source
soup = BeautifulSoup(world_html)

In [ ]:
headings = soup.find_all('span', class_ = 'mw-headline')
exclude = ['WARP ZONE', 'TWITTER']
characters = [heading.text for heading in headings if heading.text not in exclude]

We have the characters. Let's get the costumes as well. These appear as tables below the headings.

So, for each heading, we need to find the second table following the heading (the first contains details about the character)

In [ ]:
tables = {}
for heading in headings:
    if heading.find_next('table'):
        table = heading.find_next('table').find_next('table')
        if not table:
            continue

        table_headings = [t_heading.text.strip().upper() for t_heading in table.find_all('th')]
        if 'COSTUME' in table_headings:
            character = heading.text
            tables[character] = table

tables

{'Mario': <table class="fandom-table">
 <caption>
 </caption>
 <tbody><tr>
 <th>Image
 </th>
 <th>Costume
 </th>
 <th>How to Unlock
 </th>
 <th>Origin
 </th></tr>
 <tr>
 <td><span typeof="mw:File/Frameless"><a class="mw-file-description image" href="https://static.wikia.nocookie.net/mario/images/f/f1/Mario_%28Touring%29.png/revision/latest?cb=20251122000227"><img alt="Mario (Touring)" class="mw-file-element lazyload" data-image-key="Mario_%28Touring%29.png" data-image-name="Mario (Touring).png" data-relevant="1" data-src="https://static.wikia.nocookie.net/mario/images/f/f1/Mario_%28Touring%29.png/revision/latest/scale-to-width-down/121?cb=20251122000227" decoding="async" height="150" loading="lazy" src="data:image/gif;base64,R0lGODlhAQABAIABAAAAAP///yH5BAEAAAEALAAAAAABAAEAQAICTAEAOw%3D%3D" width="121"/></a></span>
 </td>
 <td>Touring
 </td>
 <td>
 </td>
 <td>
 </td></tr>
 <tr>
 <td><span typeof="mw:File/Frameless"><a class="mw-file-description image" href="https://static.wikia.nocookie

In [ ]:
all_costumes = pd.DataFrame()

for character, table in tables.items():

    table_headers = [header.text.strip() for header in table.find_all('th')]
    rows = [row for row in table.find_all('tr')]
    processed_rows = []
    for row in rows:
        processed_values = [value.text.strip() for value in row.find_all('td')]
        processed_rows.append(processed_values)

    df = pd.DataFrame(columns=table_headers, data=processed_rows)

    df['Character'] = character

    all_costumes = pd.concat([all_costumes, df])

all_costumes = all_costumes[all_costumes['Costume'].notna()]

all_costumes = all_costumes.drop(columns=['Image', 'How to Unlock', 'Origin'])

all_costumes = all_costumes.reset_index(drop=True)

all_costumes

,Costume,Character
0,Touring,Mario
1,Pro Racer,Mario
2,Mechanic,Mario
3,Dune Rider,Mario
4,Cowboy,Mario
...,...,...
98,Explorer,Baby Daisy
99,Touring,Baby Rosalina
100,Pro Racer,Baby Rosalina
101,Sailor,Baby Rosalina


In [ ]:
# Append on the basic characters to get the full list
basic_rows = [('Normal', name) for name in characters]
basic_characters = pd.DataFrame(basic_rows, columns=all_costumes.columns)

mkw_characters = pd.concat([basic_characters, all_costumes])

mkw_characters = mkw_characters.reset_index(drop=True)

mkw_characters

,Costume,Character
0,Normal,Mario
1,Normal,Luigi
2,Normal,Peach
3,Normal,Daisy
4,Normal,Yoshi
...,...,...
148,Explorer,Baby Daisy
149,Touring,Baby Rosalina
150,Pro Racer,Baby Rosalina
151,Sailor,Baby Rosalina


In [ ]:
mkw_characters.to_csv(wd / 'data' / 'mk_world_characters.csv', index=False)

Now we will move onto Mario Kart 8 Deluxe. The fact that this version doesn't have costumes should make it easier.

In [ ]:
url = "https://mariokart.fandom.com/wiki/Mario_Kart_8_Deluxe"

driver = webdriver.Firefox()
driver.get(url)

deluxe_html = driver.page_source
soup = BeautifulSoup(deluxe_html)

driver.close()

In [40]:
# We want the parts from the "Racers" section
racer_heading = soup.find_all(id="Racers")
next_heading = racer_heading[0].find_next("h2").text
next_heading

'Vehicle Parts[]'

In [31]:
import re

{'style': 'font-weight: 700; color: var(--fandom-text-color);'}

In [50]:
following_spans = racer_heading[0].find_all_next("span")

mk8_characters = []

# For each span, check the next heading and the previous sub-heading
for span in following_spans:
    if "style" in span.attrs:
        if "font-weight: 700;" not in span.attrs["style"]:
            continue
    else:
        continue

    heading_after_this = span.find_next("h2").text
    prev_section = span.find_previous("h3").text
    # If it is different to the one after the Racers heading, or we're in the Mii suit section, stop
    if heading_after_this != next_heading or prev_section == "Mii Suits[]":
        break
    else:
        mk8_characters.append(span.text)

print(mk8_characters)
print(f"There are {len(mk8_characters)} characters found")


['Mario', 'Luigi', 'Peach', 'Daisy', 'Rosalina', 'Yoshi', 'Toad', 'Koopa', 'Shy Guy', 'Lakitu', 'Toadette', 'Bowser', 'Donkey Kong', 'Wario', 'Waluigi', 'Lemmy', 'Larry', 'Wendy', 'Ludwig', 'Iggy', 'Roy', 'Morton', 'Mii', 'Tanooki Mario', 'Cat Peach', 'Baby Mario', 'Baby Luigi', 'Baby Peach', 'Baby Daisy', 'Baby Rosalina', 'Metal Mario', 'Pink Gold Peach', 'Link', 'Villager', 'Isabelle', 'King Boo', 'Dry Bones', 'Bowser Jr', 'Inkling', 'Birdo', 'Petey Piranha', 'Wiggler', 'Kamek', 'Diddy Kong', 'Funky Kong', 'Pauline', 'Peachette']
There are 47 characters found


In [ ]:
# Inkling and Villager have Male and Female versions, so we'll do a quick replacement for those
mk8_characters.remove('Inkling')
mk8_characters.remove('Villager')

mk8_characters.append('Inkling Boy')
mk8_characters.append('Inkling Girl')
mk8_characters.append('Villager (M)')
mk8_characters.append('Villager (F)')

print(f"There are {len(mk8_characters)} characters found")

There are 49 characters found


In [53]:
# Also missing Dry Bowser
mk8_characters.append('Dry Bowser')

In [55]:
# Save as a CSV
mk8_character_df = pd.DataFrame(data=mk8_characters, columns=["Character"])
mk8_character_df.to_csv(wd / 'data' / 'mk_8_characters.csv', index=False)

Now that we have all the characters for both games, let's compile them into a dataframe that contains all the appropriate information. 

The character table contains these columns:
* `name`
* `game_version`

In [3]:
mkw_characters = pd.read_csv(wd / 'data' / 'mk_world_characters.csv')
mk8_characters = pd.read_csv(wd / 'data' / 'mk_8_characters.csv')

In [59]:
mkw_characters.columns

Index(['Costume', 'Character', 'Game'], dtype='str')

In [4]:
# Add a column with the game version to each dataframe
mk8_characters['Game'] = 'Mario Kart 8 Deluxe'
mkw_characters['Game'] = 'Mario Kart World'

# Combine the costume and character columns in World
mkw_characters['Character'] = mkw_characters.apply(lambda row: " ".join([row["Costume"], row["Character"]]) if row["Costume"] != "Normal" else row["Character"], axis=1)
mkw_characters = mkw_characters.drop(columns=["Costume"])

# Concat the two dataframes
all_characters = pd.concat([mk8_characters, mkw_characters])

all_characters

,Character,Game
0,Mario,Mario Kart 8 Deluxe
1,Luigi,Mario Kart 8 Deluxe
2,Peach,Mario Kart 8 Deluxe
3,Daisy,Mario Kart 8 Deluxe
4,Rosalina,Mario Kart 8 Deluxe
...,...,...
148,Explorer Baby Daisy,Mario Kart World
149,Touring Baby Rosalina,Mario Kart World
150,Pro Racer Baby Rosalina,Mario Kart World
151,Sailor Baby Rosalina,Mario Kart World


In [5]:
# Save all characters to CSV
all_characters.to_csv(wd / "data" / "characters.csv", index=False)